In [1]:
#!/usr/bin/env python
# coding: utf-8

# ============================================================
# 特征筛选配置生成器
# 功能: 基于相关性阈值识别冗余特征，生成 feature_selection.json
#       供 data_splitting.py 读取，构建删除冗余特征后的数据集
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 加载数据
# ============================================================

df = pd.read_csv('data/full_dataset.csv')
print(f"数据形状: {df.shape}")

feature_cols = [
    'T_O', 'T_C', 'T_Ti1', 'T_Ti2',
    'Rad', 'EN',
    'Layer', 'D_E', 'D_C', 'D_M1', 'D_M2',
    'N_O', 'N_C', 'N_Ti1', 'N_Ti2', 'M_Inf',
    'SC_a', 'SC_b', 'H_A', 'H_P', 'AR', 'H_Dens', 'Sides', 'Del'
]

target_col = 'Q_M'

print(f"特征数量: {len(feature_cols)}")
print(f"目标变量: {target_col}")

# ============================================================
# 计算相关性矩阵
# ============================================================

numeric_cols = feature_cols + [target_col]
correlation_matrix = df[numeric_cols].corr()
feature_corr = correlation_matrix.loc[feature_cols, feature_cols]

# ============================================================
# 识别冗余特征（|r| > 0.9）
# ============================================================

THRESHOLD = 0.9

redundant_pairs = []
features_to_remove = set()

for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        feat1 = feature_cols[i]
        feat2 = feature_cols[j]
        corr_value = feature_corr.loc[feat1, feat2]

        if abs(corr_value) > THRESHOLD:
            corr_with_target_1 = abs(correlation_matrix.loc[feat1, target_col])
            corr_with_target_2 = abs(correlation_matrix.loc[feat2, target_col])

            # 保留与目标相关性更强的特征，删除较弱的
            if corr_with_target_1 >= corr_with_target_2:
                features_to_remove.add(feat2)
                keep_feat, remove_feat = feat1, feat2
            else:
                features_to_remove.add(feat1)
                keep_feat, remove_feat = feat2, feat1

            redundant_pairs.append({
                'Feature_1': feat1,
                'Feature_2': feat2,
                'Correlation': float(corr_value),
                'Corr_with_Target_1': float(corr_with_target_1),
                'Corr_with_Target_2': float(corr_with_target_2),
                'Kept': keep_feat,
                'Removed': remove_feat
            })

# ============================================================
# 输出筛选结果
# ============================================================

if redundant_pairs:
    print(f"\n发现 {len(redundant_pairs)} 对高度相关的特征 (|r| > {THRESHOLD}):")
    for pair in redundant_pairs:
        print(f"  {pair['Feature_1']:15s} <-> {pair['Feature_2']:15s}: "
              f"r={pair['Correlation']:+.4f}  ->  删除: {pair['Removed']}")
else:
    print(f"\n未发现冗余特征对 (|r| > {THRESHOLD})")

selected_features = [f for f in feature_cols if f not in features_to_remove]

print(f"\n原始特征数: {len(feature_cols)}")
print(f"删除特征数: {len(features_to_remove)}")
print(f"保留特征数: {len(selected_features)}")
print(f"保留比例:   {len(selected_features)/len(feature_cols)*100:.1f}%")

if features_to_remove:
    print(f"\n删除的特征:")
    for feat in sorted(features_to_remove):
        tc = correlation_matrix.loc[feat, target_col]
        print(f"  {feat:15s} (与目标相关性: {tc:+.4f})")

print(f"\n保留的特征:")
for i, feat in enumerate(selected_features, 1):
    tc = correlation_matrix.loc[feat, target_col]
    print(f"  {i:2d}. {feat:15s} (与目标相关性: {tc:+.4f})")

# ============================================================
# 保存 feature_selection.json
# ============================================================

feature_selection_results = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'method': 'correlation_threshold',
    'threshold': THRESHOLD,
    'original_features': feature_cols,
    'selected_features': selected_features,
    'removed_features': sorted(list(features_to_remove)),
    'n_original': len(feature_cols),
    'n_selected': len(selected_features),
    'n_removed': len(features_to_remove),
    'target_col': target_col,
    'redundant_pairs': redundant_pairs
}

output_path = Path('data/feature_selection.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(feature_selection_results, f, indent=4, ensure_ascii=False)

print(f"\n已保存: {output_path}")
print("data_splitting.py 将读取此文件构建删除冗余特征后的数据集。")

数据形状: (21665, 27)
特征数量: 24
目标变量: Q_M

发现 13 对高度相关的特征 (|r| > 0.9):
  T_C             <-> N_Ti1          : r=+0.9943  ->  删除: N_Ti1
  Rad             <-> EN             : r=-0.9389  ->  删除: EN
  Rad             <-> N_Ti2          : r=-0.9870  ->  删除: Rad
  D_E             <-> D_C            : r=+0.9634  ->  删除: D_C
  D_E             <-> D_M1           : r=+0.9634  ->  删除: D_M1
  D_E             <-> D_M2           : r=+0.9634  ->  删除: D_M2
  D_C             <-> D_M1           : r=+1.0000  ->  删除: D_C
  D_C             <-> D_M2           : r=+1.0000  ->  删除: D_C
  D_M1            <-> D_M2           : r=+1.0000  ->  删除: D_M1
  SC_a            <-> SC_b           : r=+0.9202  ->  删除: SC_b
  H_A             <-> H_P            : r=+0.9592  ->  删除: H_A
  H_A             <-> Del            : r=+0.9824  ->  删除: Del
  H_P             <-> Del            : r=+0.9042  ->  删除: Del

原始特征数: 24
删除特征数: 9
保留特征数: 15
保留比例:   62.5%

删除的特征:
  D_C             (与目标相关性: +0.0023)
  D_M1            (与目标相关性: +0.0023)